# CineIQ — Sentiment-Aware Re-Ranking

Audience reception matters — a movie might be popular but poorly received, or niche but beloved.
We add a **sentiment re-ranking layer** to adjust recommendation scores based on audience sentiment signals.

## Approach
Since MovieLens 1M has no review text, we derive a **sentiment proxy from ratings**:
- Rating >= 4 -> Positive (+1.0)
- Rating <= 2 -> Negative (-1.0)
- Rating = 3 -> Neutral (0.0)

We then compute the weighted sentiment per movie (adjusted by review count for confidence) and normalize to [0, 1].

In [ ]:
# Imports
import pandas as pd
import numpy as np
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pickle

# Load
movies = pd.read_csv('../data/processed/movies.csv')
ratings = pd.read_csv('../data/processed/merged.csv')

## Loading Data
We load movies and merged ratings from the processed data directory.

In [ ]:
# Simulate sentiment from ratings
# We don't have review text in ML-1M, so we derive a sentiment proxy from ratings
# Rating >= 4 = positive, <= 2 = negative, 3 = neutral
def rating_to_sentiment(rating):
    if rating >= 4:
        return 1.0
    elif rating <= 2:
        return -1.0
    else:
        return 0.0

ratings['sentiment'] = ratings['rating'].apply(rating_to_sentiment)

## Deriving Sentiment from Ratings
The `rating_to_sentiment` function maps each rating to a sentiment polarity.

In [ ]:
# Aggregate sentiment per movie
movie_sentiment = ratings.groupby('movieId').agg(
    avg_sentiment=('sentiment', 'mean'),
    review_count=('rating', 'count')
).reset_index()

# Confidence weight — more reviews = more reliable
movie_sentiment['weighted_sentiment'] = (
    movie_sentiment['avg_sentiment'] * 
    np.log1p(movie_sentiment['review_count'])
)

# Normalize to 0-1
min_s = movie_sentiment['weighted_sentiment'].min()
max_s = movie_sentiment['weighted_sentiment'].max()
movie_sentiment['sentiment_score'] = (
    (movie_sentiment['weighted_sentiment'] - min_s) / (max_s - min_s)
)

print(movie_sentiment.head())

## Aggregating Movie Sentiment
For each movie, we compute:
- **avg_sentiment**: Mean polarity across all ratings
- **review_count**: Number of ratings (confidence weight)
- **weighted_sentiment**: avg_sentiment * log1p(review_count)
- **sentiment_score**: Min-max normalized to [0, 1]

This gives higher confidence to movies with more reviews.

In [ ]:
# Reranker function
def sentiment_rerank(recommendations_df, alpha=0.3):
    """
    recommendations_df: output from ensemble_recommend (has 'movieId','title','genres','score')
    alpha: how much sentiment influences final score (0=ignore, 1=only sentiment)
    """
    merged = recommendations_df.merge(
        movie_sentiment[['movieId', 'sentiment_score']], on='movieId', how='left'
    )
    
    merged['sentiment_score'] = merged['sentiment_score'].fillna(0.5)
    merged['final_score'] = (
        (1 - alpha) * merged['score'] + alpha * merged['sentiment_score']
    )
    
    return merged[['title', 'genres', 'score', 'sentiment_score', 'final_score']]\
           .sort_values('final_score', ascending=False)

## Re-Ranker Function
`sentiment_rerank(recommendations_df, alpha=0.3)` takes the ensemble output and blends the sentiment score:

`final_score = (1 - alpha) * ensemble_score + alpha * sentiment_score`

A default alpha of 0.3 means sentiment influences 30% of the final ranking.

## End-to-End Test

Instead of mock data, we now call the actual `ensemble_recommend` function and re-rank its output — demonstrating the complete pipeline from user input to sentiment-aware recommendations.

In [ ]:
# End-to-end: ensemble -> sentiment re-rerank
import sys
sys.path.append('..')
import pickle
import json

# Load pre-trained artifacts for ensemble
cosine_sim = pickle.load(open('../models/cosine_sim.pkl', 'rb'))
indices = pickle.load(open('../models/indices.pkl', 'rb'))
svd = pickle.load(open('../models/svd_model.pkl', 'rb'))
weights = json.load(open('../models/ensemble_weights.json'))

# Popularity scores
popularity = ratings.groupby('movieId')['rating'].mean().to_dict()

# ---- Ensemble functions ----
def content_scores(title, n=50):
    if title not in indices:
        return {}
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    return {movies.iloc[i[0]]['movieId']: i[1] for i in sim_scores}

def svd_scores(user_id, movie_ids):
    return {mid: svd.predict(user_id, mid).est for mid in movie_ids}

def popularity_scores(movie_ids):
    max_r = max(popularity.values())
    return {mid: popularity.get(mid, 0) / max_r for mid in movie_ids}

def ensemble_recommend(user_id, liked_movie_title, n=10):
    candidates = content_scores(liked_movie_title, n=50)
    if not candidates:
        return None
    movie_ids = list(candidates.keys())
    max_c = max(candidates.values()) or 1
    c_scores = {mid: v/max_c for mid, v in candidates.items()}
    s_raw = svd_scores(user_id, movie_ids)
    max_s = max(s_raw.values()) or 1
    s_scores = {mid: v/max_s for mid, v in s_raw.items()}
    p_scores = popularity_scores(movie_ids)
    final = {}
    for mid in movie_ids:
        w = weights
        final[mid] = (w['w_content'] * c_scores.get(mid, 0) +
                      w['w_svd']     * s_scores.get(mid, 0) +
                      w['w_pop']     * p_scores.get(mid, 0))
    top_ids = sorted(final, key=final.get, reverse=True)[:n]
    result = movies[movies['movieId'].isin(top_ids)][['movieId','title','genres']].copy()
    result['score'] = result['movieId'].map(final)
    return result.sort_values('score', ascending=False)

# ---- Run it ----
ensemble_output = ensemble_recommend(user_id=1, liked_movie_title="Toy Story (1995)", n=10)
print("=== Ensemble Output ===")
print(ensemble_output[['title', 'genres', 'score']])
print()

# ---- Re-rank with sentiment ----
reranked = sentiment_rerank(ensemble_output)
print("=== After Sentiment Re-Ranking ===")
print(reranked)